In [1]:
from langchain_groq import ChatGroq
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from IPython.display import Image, display

In [ ]:
groq_api_key = 'gsk'
llm = ChatGroq(groq_api_key=groq_api_key, model_name="Gemma2-9b-It")

In [3]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    sentiment: str

# 1. Preprocessing Node
def preprocess(state: State) -> State:
    cleaned = state["messages"][-1].content.strip()
    state["messages"][-1].content = cleaned
    return state

# 2. Sentiment Analysis Node
def analyze_sentiment(state: State) -> State:
    msg = state["messages"][-1].content
    state["sentiment"] = "positive" if "good" in msg else "neutral"
    return state

# 3. Chatbot Node
def chatbot(state: State) -> State:
    return {"messages": llm.invoke(state['messages'])}

# 4. Logging Node
def logger(state: State) -> State:
    print(f"LOG: {state['messages'][-1].content}, Sentiment: {state.get('sentiment')}")
    return state

In [4]:
# Build the graph
builder = StateGraph(State)
builder.add_node("preprocess", preprocess)
builder.add_node("analyze_sentiment", analyze_sentiment)  # renamed
builder.add_node("chatbot", chatbot)
builder.add_node("logger", logger)

In [5]:
# Define flow
builder.add_edge(START, "preprocess")
builder.add_edge("preprocess", "analyze_sentiment")  # renamed
builder.add_edge("analyze_sentiment", "chatbot")     # renamed
builder.add_edge("chatbot", "logger")
builder.add_edge("logger", END)

In [6]:
graph = builder.compile()

In [7]:
# Draw the graph
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [22]:
# Invoke the graph
input_message = "This is a good test message."
final_state = graph.invoke({"messages": ("user", input_message)})

# Print the final state
print("Final State:", final_state)
print("Chatbot's response:", final_state['messages'][-1].content)
print("Detected Sentiment:", final_state['sentiment'])

input_message_2 = "This is a neutral message."
final_state_2 = graph.invoke({"messages": ("user", input_message_2)})
print("Final State 2:", final_state_2)
print("Chatbot's response 2:", final_state_2['messages'][-1].content)
print("Detected Sentiment 2:", final_state_2['sentiment'])

LOG: It looks like you're sending a test message. Is there anything else I can help you with, or would you like to proceed with the test?, Sentiment: positive
Final State: {'messages': [HumanMessage(content='This is a good test message.', id='611b1208-ec88-4f3a-8802-655094c59535'), AIMessage(content="It looks like you're sending a test message. Is there anything else I can help you with, or would you like to proceed with the test?", response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 42, 'total_tokens': 73, 'completion_time': 0.045471639, 'completion_tokens_details': None, 'prompt_time': 0.002159661, 'prompt_tokens_details': None, 'queue_time': 0.049141989, 'total_time': 0.0476313}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'finish_reason': 'stop', 'logprobs': None}, id='run-06c8b67e-61e0-4721-8719-96a156cb5590-0')], 'sentiment': 'positive'}
Chatbot's response: It looks like you're sending a test message. Is there anything els

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "gsk"


In [17]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",  # ✅ use a current model
    temperature=0.2,
)


In [18]:
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",  # ✅ updated model
    messages=[{"role": "user", "content": "Hello"}],
)


In [19]:
final_state = graph.invoke({"messages": ("user", input_message)})


LOG: It looks like we're starting a conversation. Is there anything specific you'd like to talk about or ask?, Sentiment: positive


In [20]:
final_state = graph.invoke({
    "messages": [("user", input_message)]
})
# or, if using LC Messages:
from langchain_core.messages import HumanMessage

final_state = graph.invoke({
    "messages": [HumanMessage(content=input_message)]
})


LOG: It looks like you're testing the chat functionality. If you have any questions or need assistance with something, feel free to ask., Sentiment: positive
LOG: It seems like a simple and straightforward message. Is there anything else I can help you with?, Sentiment: positive


In [21]:
input_message = "This is a good test message."
final_state = graph.invoke({"messages": [("user", input_message)]})

print("Final State:", final_state)
print("Chatbot's response:", final_state["messages"][-1].content)
print("Detected Sentiment:", final_state["sentiment"])

input_message_2 = "This is a neutral message."
final_state_2 = graph.invoke({"messages": [("user", input_message_2)]})

print("Final State 2:", final_state_2)
print("Chatbot's response 2:", final_state_2["messages"][-1].content)
print("Detected Sentiment 2:", final_state_2["sentiment"])


LOG: It looks like you're testing the chat functionality. If you need help or have questions, feel free to ask., Sentiment: positive
Final State: {'messages': [HumanMessage(content='This is a good test message.', id='04cd04fa-4095-4dde-bb93-a89981277346'), AIMessage(content="It looks like you're testing the chat functionality. If you need help or have questions, feel free to ask.", response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 42, 'total_tokens': 66, 'completion_time': 0.041470405, 'completion_tokens_details': None, 'prompt_time': 0.001895267, 'prompt_tokens_details': None, 'queue_time': 0.049968762, 'total_time': 0.043365672}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'finish_reason': 'stop', 'logprobs': None}, id='run-2c90e42c-1bee-4c96-91a8-4f0c59a0a73f-0')], 'sentiment': 'positive'}
Chatbot's response: It looks like you're testing the chat functionality. If you need help or have questions, feel free to ask.
Detected 